In [1]:
# 학습된 모델을 테스트 해볼 수 있는 코드

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install -U qwen-tts

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.4/61.4 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 8.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.5/113.5 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 120.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 102.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 57.4 MB/s eta 0:00:00
  Created wheel for sox: filename=sox-1.5.0-py3-none-any.whl size=40036 sha256=10733901c76b622fd77fb83f2f4804ea997301b4abf6baa6490c4530451e4947
  Stored in directory: /root/.cache/pip/wheels/8c/c7/e7/baea1f7e79b9eb53addc81cc9b827424f4a7d8c9cc18c03659
Successfully built sox
  Attempting uninstall: huggingface_hub
  

In [4]:
!apt-get update -qq
!apt-get install -y sox libsox-fmt-all

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libao-common libao4 libid3tag0 libmad0 libopencore-amrnb0 libopencore-amrwb0
  libsox-fmt-alsa libsox-fmt-ao libsox-fmt-base libsox-fmt-mp3 libsox-fmt-oss
  libsox-fmt-pulse libsox3 libwavpack1
Suggested packages:
  libaudio2 libsndio6.1
The following NEW packages will be installed:
  libao-common libao4 libid3tag0 libmad0 libopencore-amrnb0 libopencore-amrwb0
  libsox-fmt-all libsox-fmt-alsa libsox-fmt-ao libsox-fmt-base libsox-fmt-mp3
  libsox-fmt-oss libsox-fmt-pulse libsox3 libwavpack1 sox
0 upgraded, 16 newly installed, 0 to remove and 106 not upgraded.
Need to get 800 kB of archives.
After this operation, 2,533 kB of additional disk space will b

In [10]:
import os
os.environ["TRANSFORMERS_NO_TF"] = "1"

from pathlib import Path
import torch
import soundfile as sf
from qwen_tts import Qwen3TTSModel
from IPython.display import Audio, display

# =========================
# 설정
# =========================
SPEAKER = "jhc100"
BASE_DIR = Path("/content/drive/MyDrive/비트메이트_TP01")
CKPT_ROOT = BASE_DIR / "qwen3_ft_output" / SPEAKER

SELECTED_EPOCH = None  # None이면 최신

# =========================
# 최신 epoch 자동 선택
# =========================
if SELECTED_EPOCH is None:
    epochs = [
        int(p.name.split("-")[-1])
        for p in CKPT_ROOT.iterdir()
        if p.is_dir() and p.name.startswith("checkpoint-epoch-")
    ]

    if not epochs:
        raise RuntimeError(f"checkpoint 없음: {CKPT_ROOT}")

    SELECTED_EPOCH = max(epochs)

print("선택 epoch:", SELECTED_EPOCH)

# =========================
# 체크포인트 경로
# =========================
target_ckpt = CKPT_ROOT / f"checkpoint-epoch-{SELECTED_EPOCH}"

if not target_ckpt.exists():
    raise FileNotFoundError(f"해당 epoch 없음: {target_ckpt}")

weight_file = target_ckpt / "model.safetensors"
if not weight_file.exists():
    raise RuntimeError(f"가중치 없음: {target_ckpt}")

print("선택된 체크포인트:", target_ckpt)

# =========================
# 모델 로드
# =========================
tts = Qwen3TTSModel.from_pretrained(
    str(target_ckpt),
    device_map="cuda:0" if torch.cuda.is_available() else "cpu",
    dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
)

# =========================
# 테스트 텍스트
# =========================
texts = [
    "안녕하세요, 오늘 하루도 차분하게 시작해 보겠습니다.",
    "오늘 하루도 차분하게 시작해.",
    "와, 생각보다 훨씬 자연스럽게 들리네요!",
    "괜찮아요, 천천히 다시 해보면 됩니다.",
    "야 너 왜 그렇게 사냐",
    "개 짖는 소리 좀 안나게 해라",
    "This is my TTS, bro",
    "1,2,3,4,5",
    "010-1234-5677",
]

# =========================
# 생성
# =========================
wavs, sr = tts.generate_custom_voice(
    text=texts,
    speaker=SPEAKER,
    # instruct="happy"   # 감정 컨트롤
)

# =========================
# 저장 & 재생
# =========================
OUTPUT_DIR = Path("/content")
OUTPUT_DIR.mkdir(exist_ok=True)

for i, wav in enumerate(wavs):
    out_path = OUTPUT_DIR / f"{SPEAKER}_epoch{SELECTED_EPOCH}_{i+1}.wav"

    sf.write(str(out_path), wav, sr)

    print(f"\n===== 문장 {i+1} =====")
    print(texts[i])
    print("저장:", out_path)

    display(Audio(str(out_path)))

선택 epoch: 3
선택된 체크포인트: /content/drive/MyDrive/비트메이트_TP01/qwen3_ft_output/jhc100/checkpoint-epoch-3


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.



===== 문장 1 =====
안녕하세요, 오늘 하루도 차분하게 시작해 보겠습니다.
저장: /content/jhc100_epoch3_1.wav



===== 문장 2 =====
오늘 하루도 차분하게 시작해.
저장: /content/jhc100_epoch3_2.wav



===== 문장 3 =====
와, 생각보다 훨씬 자연스럽게 들리네요!
저장: /content/jhc100_epoch3_3.wav


...


===== 문장 5 =====
야 너 왜 그렇게 사냐
저장: /content/jhc100_epoch3_5.wav



===== 문장 6 =====
개 짖는 소리 좀 안나게 해라
저장: /content/jhc100_epoch3_6.wav



===== 문장 7 =====
This is my TTS, bro
저장: /content/jhc100_epoch3_7.wav



===== 문장 8 =====
1,2,3,4,5
저장: /content/jhc100_epoch3_8.wav



===== 문장 9 =====
010-1234-5677
저장: /content/jhc100_epoch3_9.wav
